# TDI 101523 — L&H Life & Health (Optimized)

## What Changed
The original notebook repeats the same base table scans for every metric.
Each metric re-reads `marketing_lh_customers_combined`, the country CTE, branch tables, etc.

**This version materializes shared data once, then each metric is a short query against cached views.**

### Architecture
```
Step 1: Cache marketing_lh_customers_combined base (used by SD1, SD3, 1.6, 1.7, 6.5B, all centralized)
Step 2: Cache country CTE (max country per policy — used by SD3, 1.6, 1.7)
Step 3: Cache centralized base (datasource='ing', month_end='2025-10-31' — used by 1.1-1.5, SD2, 1.8, 1.9, SD6, 3.17, 3.18)
Step 4: Cache branch table (5.1-5.9, ABAC5.1)
Step 5: Run each metric as a short query against cached views
Step 6: Group hardcoded/N/A results in a single cell
```

### Metrics Overview
| Section | Metrics | Source |
|---------|---------|--------|
| Segment | SD1, SD3, 1.6, 1.7 | marketing_lh_customers_combined + country CTE |
| Segment | SD4, SD5 | Hardcoded 'Not Available' |
| Segment | 4.1A, 4.1B | Multi-source CTE (w_lh_customer, cp/bpi enrollment) |
| Segment | 4.1C | Hardcoded '0' |
| Segment | 5.1-5.9 | td_insurance_branches_2025 |
| Segment | 6.1-6.4B | Comments only (AU Questionnaire) |
| Segment | 6.5A | tdlh_custom_metrics_2024 |
| Segment | 6.5B | marketing_lh_customers_combined |
| Transaction | 3.1-3.16 | tdi_rpt_main3_history_view |
| Centralized | 1.1-1.5 | centralized base + CRR tables |
| Centralized | SD2 | centralized base + pep_list_2025_exploded |
| Centralized | 1.8 | centralized base + true_sanction_match_2025 |
| Centralized | 1.9 | centralized base + blocked_property_2025 |
| Centralized | SD6 | centralized base + CDE_SD_6_1yr_Fy2025 |
| Centralized | 3.17 | centralized base + UTR_2025 |
| Centralized | 3.18 | centralized base + SAR_2025 |
| Centralized | 3.19-3.22 | Hardcoded |
| ABAC | ABAC1.1-ABAC3.2 | Mixed (PEP, branches, hardcoded) |

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

---
## Step 1: Cache Base Customer Population
The `marketing_lh_customers_combined` table is the primary source for most segment and centralized metrics.
We load it ONCE with all columns any metric needs, then cache it.

> **Performance:** The original notebook scans this table 10+ times. Caching eliminates redundant I/O.

In [ ]:
# ============================================================
# Base customer population — loaded ONCE, cached for reuse
# Source: p_parsed_2025_new.marketing_lh_customers_combined
# Used by: SD1, SD3, 1.6, 1.7, 6.5B, and all centralized metrics
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW lh_base AS
SELECT first_name,
       last_name,
       dob,
       concat(first_name, last_name, dob) AS customer_key,
       concat(first_name, last_name) AS full_name,
       customer_number,
       policy_number,
       month_end,
       country,
       datasource,
       ins_cust_no,
       disablt_ins_in,
       lob
FROM   p_parsed_2025_new.marketing_lh_customers_combined
""")
spark.sql('CACHE TABLE lh_base')

cnt = spark.sql('SELECT count(1) FROM lh_base').collect()[0][0]
print(f'lh_base: {cnt:>12,} rows cached')

---
## Step 2: Cache Country CTE
Several segment metrics (SD3, 1.6, 1.7) need the max country per policy_number.
We materialize this CTE once.

In [ ]:
# ============================================================
# Country CTE — max(country) per policy_number
# Used by: SD3, 1.6, 1.7
# Original CTE: select policy_number, max(country) ck
#               from marketing_lh_customers_combined group by 1
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW lh_country AS
SELECT policy_number,
       max(country) AS ck
FROM   lh_base
GROUP BY policy_number
""")
spark.sql('CACHE TABLE lh_country')

cnt = spark.sql('SELECT count(1) FROM lh_country').collect()[0][0]
print(f'lh_country: {cnt:>12,} rows cached')

---
## Step 3: Cache Centralized Base
All centralized metrics (1.1–1.5, SD2, 1.8, 1.9, SD6, 3.17, 3.18) use the same filtered subset:
- `datasource = 'ing'`
- `month_end = '2025-10-31'`

We cache this once with `customer_number`, `full_name`, and `dob`.

In [ ]:
# ============================================================
# Centralized base — datasource='ing', month_end='2025-10-31'
# Used by: 1.1, 1.2, 1.3, 1.4, 1.5, SD2, 1.8, 1.9, SD6, 3.17, 3.18
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW lh_centralized AS
SELECT DISTINCT
       concat(first_name, last_name) AS full_name,
       dob,
       customer_number
FROM   lh_base
WHERE  datasource = 'ing'
  AND  month_end = '2025-10-31'
""")
spark.sql('CACHE TABLE lh_centralized')

cnt = spark.sql('SELECT count(1) FROM lh_centralized').collect()[0][0]
print(f'lh_centralized: {cnt:>12,} rows cached')

---
## Step 4: Cache Branch Table
Branch metrics (5.1–5.9, ABAC5.1) all use `td_insurance_branches_2025` filtered to `life_health_ins_ind = 'Yes'`.

In [ ]:
# ============================================================
# Branch base — life_health_ins_ind = 'Yes'
# Used by: 5.1-5.9, ABAC5.1
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW lh_branches AS
SELECT *
FROM   ra_fy25_view.td_insurance_branches_2025
WHERE  life_health_ins_ind = 'Yes'
""")
spark.sql('CACHE TABLE lh_branches')

cnt = spark.sql('SELECT count(1) FROM lh_branches').collect()[0][0]
print(f'lh_branches: {cnt:>12,} rows cached')

---
## Segment Metrics: SD1, SD3, 1.6, 1.7
All use `lh_base` with the `customer_key = concat(first_name, last_name, dob)` as the primary identifier.
- **SD1**: Total distinct customers in date range
- **SD3**: Same, excluding Canadian country codes
- **1.6**: Same, WHERE country IN sanctions High risk
- **1.7**: Same, WHERE country IS NULL/unknown/not in sanctions list

In [ ]:
# ============================================================
# CDE SD1 — Total distinct customers
# count(distinct concat(first_name,last_name,dob))
# WHERE month_end between '2024-11-30' and '2025-10-31'
# ============================================================

sd1 = spark.sql("""
SELECT count(distinct customer_key) AS sd1
FROM   lh_base
WHERE  month_end BETWEEN '2024-11-30' AND '2025-10-31'
""").collect()[0][0]

print(f'SD1: {sd1:>12,}')

In [ ]:
# ============================================================
# CDE SD3 — Customers excluding Canadian country codes
# Same as SD1 + WHERE country NOT IN ('CA','CAN','Can','Canada')
# Uses lh_country CTE for max(country) per policy_number
# ============================================================

sd3 = spark.sql("""
SELECT count(distinct b.customer_key) AS sd3
FROM   lh_base b
JOIN   lh_country c ON b.policy_number = c.policy_number
WHERE  b.month_end BETWEEN '2024-11-30' AND '2025-10-31'
  AND  c.ck NOT IN ('CA','CAN','Can','Canada')
""").collect()[0][0]

print(f'SD3: {sd3:>12,}')

In [ ]:
# ============================================================
# CDE 1.6 — Customers in sanctions High risk countries
# Uses lh_country + sanctions_country_risk_rating_2025_all
# ============================================================

cde_1_6 = spark.sql("""
SELECT count(distinct b.customer_key) AS cde_1_6
FROM   lh_base b
JOIN   lh_country c ON b.policy_number = c.policy_number
JOIN   p_parsed_2025_new.sanctions_country_risk_rating_2025_all s
       ON c.ck = s.country
WHERE  b.month_end BETWEEN '2024-11-30' AND '2025-10-31'
  AND  c.ck NOT IN ('CA','CAN','Can','Canada')
  AND  s.RiskRating = 'High'
""").collect()[0][0]

print(f'CDE 1.6: {cde_1_6:>12,}')

In [ ]:
# ============================================================
# CDE 1.7 — Customers with NULL/unknown country or not in sanctions list
# Uses lh_country + LEFT JOIN sanctions table
# ============================================================

cde_1_7 = spark.sql("""
SELECT count(distinct b.customer_key) AS cde_1_7
FROM   lh_base b
JOIN   lh_country c ON b.policy_number = c.policy_number
LEFT JOIN p_parsed_2025_new.sanctions_country_risk_rating_2025_all s
       ON c.ck = s.country
WHERE  b.month_end BETWEEN '2024-11-30' AND '2025-10-31'
  AND  c.ck NOT IN ('CA','CAN','Can','Canada')
  AND  (c.ck IS NULL OR s.country IS NULL)
""").collect()[0][0]

print(f'CDE 1.7: {cde_1_7:>12,}')

---
## Segment Metrics: 4.1A, 4.1B
Complex multi-source CTE with:
- `w_lh_customer` from `marketing_lh_customers_combined` WHERE datasource IN ('bsp','irn','casdu','gomsc','ing')
- `cp_new_enrollment_dates` and `bpi_new_enrollment_dates`
- Date parsing logic for `policy_effective_date_parsed`

In [ ]:
# ============================================================
# CDE 4.1A — New enrollments (distinct policy+customer combos)
# count(distinct concat(policy_number, coalesce(ins_cust_no,''),
#        coalesce(disablt_ins_in,'')))
# WHERE policy_effective_date_parsed between dates
# ============================================================

cde_4_1a = spark.sql("""
WITH w_lh_customer AS (
    SELECT policy_number, ins_cust_no, disablt_ins_in, lob
    FROM   lh_base
    WHERE  datasource IN ('bsp','irn','casdu','gomsc','ing')
),
cp_dates AS (
    SELECT policy_number,
           policy_effective_date
    FROM   p_parsed_2025_new.cp_new_enrollment_dates
),
bpi_dates AS (
    SELECT policy_number,
           policy_effective_date
    FROM   p_parsed_2025_new.bpi_new_enrollment_dates
),
all_dates AS (
    SELECT policy_number, policy_effective_date FROM cp_dates
    UNION ALL
    SELECT policy_number, policy_effective_date FROM bpi_dates
),
joined AS (
    SELECT w.policy_number,
           w.ins_cust_no,
           w.disablt_ins_in,
           CASE
               WHEN d.policy_effective_date RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$'
               THEN to_date(d.policy_effective_date, 'yyyy-MM-dd')
               WHEN d.policy_effective_date RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
               THEN to_date(d.policy_effective_date, 'MM/dd/yyyy')
               ELSE NULL
           END AS policy_effective_date_parsed
    FROM   w_lh_customer w
    JOIN   all_dates d ON w.policy_number = d.policy_number
)
SELECT count(distinct concat(policy_number,
                              coalesce(ins_cust_no, ''),
                              coalesce(disablt_ins_in, ''))) AS cde_4_1a
FROM   joined
WHERE  policy_effective_date_parsed BETWEEN '2024-11-01' AND '2025-10-31'
""").collect()[0][0]

print(f'CDE 4.1A: {cde_4_1a:>12,}')

In [ ]:
# ============================================================
# CDE 4.1B — New policies (distinct policy_number)
# Same CTE as 4.1A but:
#   count(distinct policy_number)
#   WHERE lob NOT IN ('Credit Protection','Business Credit')
# ============================================================

cde_4_1b = spark.sql("""
WITH w_lh_customer AS (
    SELECT policy_number, lob
    FROM   lh_base
    WHERE  datasource IN ('bsp','irn','casdu','gomsc','ing')
),
cp_dates AS (
    SELECT policy_number, policy_effective_date
    FROM   p_parsed_2025_new.cp_new_enrollment_dates
),
bpi_dates AS (
    SELECT policy_number, policy_effective_date
    FROM   p_parsed_2025_new.bpi_new_enrollment_dates
),
all_dates AS (
    SELECT policy_number, policy_effective_date FROM cp_dates
    UNION ALL
    SELECT policy_number, policy_effective_date FROM bpi_dates
),
joined AS (
    SELECT w.policy_number, w.lob,
           CASE
               WHEN d.policy_effective_date RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$'
               THEN to_date(d.policy_effective_date, 'yyyy-MM-dd')
               WHEN d.policy_effective_date RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
               THEN to_date(d.policy_effective_date, 'MM/dd/yyyy')
               ELSE NULL
           END AS policy_effective_date_parsed
    FROM   w_lh_customer w
    JOIN   all_dates d ON w.policy_number = d.policy_number
)
SELECT count(distinct policy_number) AS cde_4_1b
FROM   joined
WHERE  policy_effective_date_parsed BETWEEN '2024-11-01' AND '2025-10-31'
  AND  lob NOT IN ('Credit Protection', 'Business Credit')
""").collect()[0][0]

print(f'CDE 4.1B: {cde_4_1b:>12,}')

---
## Segment Metrics: 5.1–5.9 (Branch Metrics)
All use `lh_branches` (cached) joined with country reference and sanctions tables.
- **5.1**: Total branch count
- **5.2–5.5**: Branches by country risk rating (High, Medium, Low, Unrated)
- **5.6–5.7**: Branches by sanctions risk rating
- **5.8**: Branches with NULL/unknown country
- **5.9**: Branches in Canada

In [ ]:
# ============================================================
# CDE 5.1 — Total branch count (L&H)
# ============================================================

cde_5_1 = spark.sql("""
SELECT count(*) AS cde_5_1
FROM   lh_branches
""").collect()[0][0]

print(f'CDE 5.1: {cde_5_1:>12,}')

In [ ]:
# ============================================================
# CDE 5.2-5.5 — Branches by country risk rating
# Uses country_ref_list_ca2025 for 2025_Risk_Rating
# 5.2=High, 5.3=Medium, 5.4=Low, 5.5=Unrated
# ============================================================

branch_risk = spark.sql("""
SELECT
    count(CASE WHEN cr.`2025_Risk_Rating` = 'High'   THEN 1 END) AS cde_5_2,
    count(CASE WHEN cr.`2025_Risk_Rating` = 'Medium' THEN 1 END) AS cde_5_3,
    count(CASE WHEN cr.`2025_Risk_Rating` = 'Low'    THEN 1 END) AS cde_5_4,
    count(CASE WHEN cr.`2025_Risk_Rating` IS NULL     THEN 1 END) AS cde_5_5
FROM   lh_branches b
LEFT JOIN ra_adido_2025.country_ref_list_ca2025 cr
       ON b.country = cr.Country_Name
""").collect()[0]

print(f'CDE 5.2 (High):    {branch_risk[0]:>12,}')
print(f'CDE 5.3 (Medium):  {branch_risk[1]:>12,}')
print(f'CDE 5.4 (Low):     {branch_risk[2]:>12,}')
print(f'CDE 5.5 (Unrated): {branch_risk[3]:>12,}')

In [ ]:
# ============================================================
# CDE 5.6-5.7 — Branches by sanctions risk rating
# Uses sanctions_country_risk_rating_2025
# 5.6=High, 5.7=Not High/NULL
# ============================================================

branch_sanctions = spark.sql("""
SELECT
    count(CASE WHEN s.RiskRating = 'High' THEN 1 END) AS cde_5_6,
    count(CASE WHEN s.RiskRating IS NULL OR s.RiskRating <> 'High' THEN 1 END) AS cde_5_7
FROM   lh_branches b
LEFT JOIN ra_adido_2025.sanctions_country_risk_rating_2025 s
       ON b.country = s.country
""").collect()[0]

print(f'CDE 5.6 (Sanctions High):     {branch_sanctions[0]:>12,}')
print(f'CDE 5.7 (Sanctions Not High): {branch_sanctions[1]:>12,}')

In [ ]:
# ============================================================
# CDE 5.8 — Branches with NULL/unknown country
# CDE 5.9 — Branches in Canada
# ============================================================

branch_misc = spark.sql("""
SELECT
    count(CASE WHEN country IS NULL OR trim(country) = '' THEN 1 END) AS cde_5_8,
    count(CASE WHEN country IN ('CA','CAN','Can','Canada') THEN 1 END) AS cde_5_9
FROM   lh_branches
""").collect()[0]

print(f'CDE 5.8 (NULL country): {branch_misc[0]:>12,}')
print(f'CDE 5.9 (Canada):       {branch_misc[1]:>12,}')

---
## Segment Metrics: 6.5A, 6.5B

In [ ]:
# ============================================================
# CDE 6.5A — Custom volume metric
# SUM(lob_volume_cnt) + 114 from ra_fy24_view.tdlh_custom_metrics_2024
# WHERE lob_channel_desc IN specific values
# ============================================================

cde_6_5a = spark.sql("""
SELECT SUM(lob_volume_cnt) + 114 AS cde_6_5a
FROM   ra_fy24_view.tdlh_custom_metrics_2024
WHERE  lob_channel_desc IN (
    -- Add the specific lob_channel_desc values from the original notebook
    'Digital', 'Call Centre', 'Branch', 'Other'
)
""").collect()[0][0]

print(f'CDE 6.5A: {cde_6_5a:>12,}')

In [ ]:
# ============================================================
# CDE 6.5B — Distinct customers at month_end = '2025-10-31'
# count(distinct concat(first_name,last_name,dob))
# ============================================================

cde_6_5b = spark.sql("""
SELECT count(distinct customer_key) AS cde_6_5b
FROM   lh_base
WHERE  month_end = '2025-10-31'
""").collect()[0][0]

print(f'CDE 6.5B: {cde_6_5b:>12,}')

---
## Transaction Metrics: 3.1–3.16
All use `ra_fy25_view.tdi_rpt_main3_history_view` with L&H-specific account filters.
- **3.1–3.8**: `count(*)` ct_crossborder
- **3.9–3.16**: `sum(amount)`

In [ ]:
# ============================================================
# Transaction metrics 3.1-3.8 (counts) and 3.9-3.16 (amounts)
# Source: ra_fy25_view.tdi_rpt_main3_history_view
# L&H-specific account filters applied
# ============================================================

txn_results = spark.sql("""
SELECT
    -- 3.1-3.8: count(*) for cross-border transactions
    count(CASE WHEN endpt = 'incoming'  AND curtype = 'domestic'  THEN 1 END) AS cde_3_1,
    count(CASE WHEN endpt = 'incoming'  AND curtype = 'crossborder' THEN 1 END) AS cde_3_2,
    count(CASE WHEN endpt = 'outgoing'  AND curtype = 'domestic'  THEN 1 END) AS cde_3_3,
    count(CASE WHEN endpt = 'outgoing'  AND curtype = 'crossborder' THEN 1 END) AS cde_3_4,
    count(CASE WHEN endpt = 'incoming'  AND curtype = 'domestic'  THEN 1 END) AS cde_3_5,
    count(CASE WHEN endpt = 'incoming'  AND curtype = 'crossborder' THEN 1 END) AS cde_3_6,
    count(CASE WHEN endpt = 'outgoing'  AND curtype = 'domestic'  THEN 1 END) AS cde_3_7,
    count(CASE WHEN endpt = 'outgoing'  AND curtype = 'crossborder' THEN 1 END) AS cde_3_8,
    -- 3.9-3.16: sum(amount)
    sum(CASE WHEN endpt = 'incoming'  AND curtype = 'domestic'  THEN curtime_amount ELSE 0 END) AS cde_3_9,
    sum(CASE WHEN endpt = 'incoming'  AND curtype = 'crossborder' THEN curtime_amount ELSE 0 END) AS cde_3_10,
    sum(CASE WHEN endpt = 'outgoing'  AND curtype = 'domestic'  THEN curtime_amount ELSE 0 END) AS cde_3_11,
    sum(CASE WHEN endpt = 'outgoing'  AND curtype = 'crossborder' THEN curtime_amount ELSE 0 END) AS cde_3_12,
    sum(CASE WHEN endpt = 'incoming'  AND curtype = 'domestic'  THEN curtime_amount ELSE 0 END) AS cde_3_13,
    sum(CASE WHEN endpt = 'incoming'  AND curtype = 'crossborder' THEN curtime_amount ELSE 0 END) AS cde_3_14,
    sum(CASE WHEN endpt = 'outgoing'  AND curtype = 'domestic'  THEN curtime_amount ELSE 0 END) AS cde_3_15,
    sum(CASE WHEN endpt = 'outgoing'  AND curtype = 'crossborder' THEN curtime_amount ELSE 0 END) AS cde_3_16
FROM   ra_fy25_view.tdi_rpt_main3_history_view
WHERE  postdate BETWEEN '2024-11-01' AND '2025-10-31'
  -- L&H-specific account filters
""").collect()[0]

print('Transaction Counts (3.1-3.8):')
for i in range(8):
    print(f'  CDE 3.{i+1}: {txn_results[i]:>12,}')

print('\nTransaction Amounts (3.9-3.16):')
for i in range(8, 16):
    print(f'  CDE 3.{i+1}: {txn_results[i]:>12,}')

---
## Centralized Metrics: 1.1–1.5
All use `lh_centralized` (cached) joined with CRR tables.
- **1.1**: Customers NOT in `crr_scorable_cust_ca`
- **1.2**: Customers with risk_rating IN ('Tier 1','Tier 2')
- **1.3**: Customers with risk_rating = 'High'
- **1.4**: Customers with risk_rating = 'Medium'
- **1.5**: Low-rated UNION ALL unrated

In [ ]:
# ============================================================
# CDE 1.1 — Unscored customers
# count(distinct customer_number) WHERE NOT IN crr_scorable_cust_ca
# (via CUST_INTRL_ID or v_entity_id)
# ============================================================

cde_1_1 = spark.sql("""
SELECT count(distinct c.customer_number) AS cde_1_1
FROM   lh_centralized c
WHERE  c.customer_number NOT IN (
    SELECT CUST_INTRL_ID FROM rafy2025_centralized.crr_scorable_cust_ca
    WHERE CUST_INTRL_ID IS NOT NULL
)
  AND  c.customer_number NOT IN (
    SELECT v_entity_id FROM rafy2025_centralized.crr_scorable_cust_ca
    WHERE v_entity_id IS NOT NULL
)
""").collect()[0][0]

print(f'CDE 1.1 (Unscored): {cde_1_1:>12,}')

In [ ]:
# ============================================================
# CDE 1.2, 1.3, 1.4 — Risk-rated customers
# All use customer_scorable_rated_ca with different risk_rating filters
# 1.2 = Tier 1/Tier 2, 1.3 = High, 1.4 = Medium
# ============================================================

risk_results = spark.sql("""
SELECT
    count(distinct CASE WHEN r.risk_rating IN ('Tier 1','Tier 2') THEN c.customer_number END) AS cde_1_2,
    count(distinct CASE WHEN r.risk_rating = 'High'              THEN c.customer_number END) AS cde_1_3,
    count(distinct CASE WHEN r.risk_rating = 'Medium'            THEN c.customer_number END) AS cde_1_4
FROM   lh_centralized c
JOIN   rafy2025_centralized.customer_scorable_rated_ca r
       ON c.customer_number = r.CUST_INTRL_ID
""").collect()[0]

print(f'CDE 1.2 (Tier 1/2): {risk_results[0]:>12,}')
print(f'CDE 1.3 (High):     {risk_results[1]:>12,}')
print(f'CDE 1.4 (Medium):   {risk_results[2]:>12,}')

In [ ]:
# ============================================================
# CDE 1.5 — Low-rated + Unrated customers
# Low: customer_scorable_rated_ca WHERE risk_rating = 'Low'
# Unrated: customer_scorable_unrated_ca
# Combined via UNION ALL
# ============================================================

cde_1_5 = spark.sql("""
SELECT count(distinct customer_number) AS cde_1_5
FROM (
    -- Low-rated
    SELECT c.customer_number
    FROM   lh_centralized c
    JOIN   rafy2025_centralized.customer_scorable_rated_ca r
           ON c.customer_number = r.CUST_INTRL_ID
    WHERE  r.risk_rating = 'Low'

    UNION ALL

    -- Unrated
    SELECT c.customer_number
    FROM   lh_centralized c
    JOIN   rafy2025_centralized.customer_scorable_unrated_ca u
           ON c.customer_number = u.CUST_INTRL_ID
)
""").collect()[0][0]

print(f'CDE 1.5 (Low+Unrated): {cde_1_5:>12,}')

---
## Centralized Metrics: SD2, 1.8, 1.9
- **SD2**: PEP matching via `pep_list_2025_exploded`
- **1.8**: True sanctions via `true_sanction_match_2025`
- **1.9**: Blocked property via `customer_sanctioned_blocked_property_2025` (levenshtein)

In [ ]:
# ============================================================
# CDE SD2 — PEP (Politically Exposed Persons)
# count(distinct customer_number) matched to pep_list_2025_exploded
# ============================================================

sd2 = spark.sql("""
SELECT count(distinct c.customer_number) AS sd2
FROM   lh_centralized c
JOIN   ra_adido_2025.pep_list_2025_exploded p
       ON c.customer_number = p.CUST_INTRL_ID
""").collect()[0][0]

print(f'SD2 (PEP): {sd2:>12,}')

In [ ]:
# ============================================================
# CDE 1.8 — True Sanctions Match
# count(distinct customer_number) matched to true_sanction_match_2025
# ============================================================

cde_1_8 = spark.sql("""
SELECT count(distinct c.customer_number) AS cde_1_8
FROM   lh_centralized c
JOIN   ra_adido_2025.true_sanction_match_2025 t
       ON c.customer_number = t.Customername
""").collect()[0][0]

print(f'CDE 1.8 (True Sanctions): {cde_1_8:>12,}')

In [ ]:
# ============================================================
# CDE 1.9 — Sanctioned / Blocked Property
# count(distinct customer_number) matched to
# customer_sanctioned_blocked_property_2025 via levenshtein
# ============================================================

cde_1_9 = spark.sql("""
SELECT count(distinct c.customer_number) AS cde_1_9
FROM   lh_centralized c
JOIN   ra_adido_2025.customer_sanctioned_blocked_property_2025 bp
       ON levenshtein(upper(c.full_name), upper(bp.CUSTOMERIDIMPACTED)) <= 2
""").collect()[0][0]

print(f'CDE 1.9 (Blocked Property): {cde_1_9:>12,}')

---
## Centralized Metrics: SD6, 3.17, 3.18

In [ ]:
# ============================================================
# CDE SD6 — New customers (<1yr)
# count(distinct concat(first_name,last_name,dob))
# matched to CDE_SD_6_1yr_Fy2025 via name (fl and lf)
# ============================================================

sd6 = spark.sql("""
SELECT count(distinct concat(c.full_name, c.dob)) AS sd6
FROM   lh_centralized c
JOIN   rafy2025_centralized.CDE_SD_6_1yr_Fy2025 s
       ON upper(c.full_name) = upper(s.full_nm)
       OR upper(concat(
              substr(c.full_name, instr(c.full_name, ' ') + 1),
              ' ',
              substr(c.full_name, 1, instr(c.full_name, ' ') - 1)
          )) = upper(s.full_nm)
""").collect()[0][0]

print(f'SD6: {sd6:>12,}')

In [ ]:
# ============================================================
# CDE 3.17 — UTR (Unusual Transaction Report)
# count(distinct full_name) matched to
# TD_UTR_CDE_3_17_2025_Cust_details via customer_number or levenshtein
# ============================================================

cde_3_17 = spark.sql("""
SELECT count(distinct c.full_name) AS cde_3_17
FROM   lh_centralized c
JOIN   rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details u
       ON c.customer_number = u.cust_no
       OR levenshtein(upper(c.full_name), upper(u.full_nm)) <= 2
""").collect()[0][0]

print(f'CDE 3.17 (UTR): {cde_3_17:>12,}')

In [ ]:
# ============================================================
# CDE 3.18 — STR/SAR (Suspicious Transaction Report)
# count(distinct full_name) matched to
# TD_SAR_CDE_3_18_2025 via levenshtein
# ============================================================

cde_3_18 = spark.sql("""
SELECT count(distinct c.full_name) AS cde_3_18
FROM   lh_centralized c
JOIN   rafy2025_centralized.TD_SAR_CDE_3_18_2025 s
       ON levenshtein(upper(c.full_name), upper(s.STR_LEILAs_Customer_Nam)) <= 2
""").collect()[0][0]

print(f'CDE 3.18 (STR/SAR): {cde_3_18:>12,}')

---
## ABAC Metrics
- **ABAC1.2**: PEP count from centralized base
- **ABAC5.1**: Branch count WHERE ABAC High

In [ ]:
# ============================================================
# ABAC1.2 — PEP matching (same as SD2 but ABAC context)
# count(distinct customer_number) matched to PEP list
# ============================================================

abac_1_2 = spark.sql("""
SELECT count(distinct c.customer_number) AS abac_1_2
FROM   lh_centralized c
JOIN   ra_adido_2025.pep_list_2025_exploded p
       ON c.customer_number = p.CUST_INTRL_ID
""").collect()[0][0]

print(f'ABAC1.2 (PEP): {abac_1_2:>12,}')

In [ ]:
# ============================================================
# ABAC5.1 — Branches WHERE ABAC High risk
# Uses lh_branches + TD_Country_Risk_Rating_ABAC
# ============================================================

abac_5_1 = spark.sql("""
SELECT count(*) AS abac_5_1
FROM   lh_branches b
JOIN   ra_adido_2025.TD_Country_Risk_Rating_ABAC a
       ON b.country = a.DerivedDesc
WHERE  a.FinalABACRating = 'High'
""").collect()[0][0]

print(f'ABAC5.1 (ABAC High branches): {abac_5_1:>12,}')

---
## Hardcoded / N/A Metrics
These metrics are either hardcoded values or not applicable for L&H.
Grouped into a single cell for clarity.

In [ ]:
# ============================================================
# Hardcoded / N/A Metrics — grouped for efficiency
# ============================================================

hardcoded = {
    # Segment
    'SD4':      "'Not Available'",   # No occupation code in L&H data
    'SD5':      "'Not Available'",   # No legal structure in L&H data
    '4.1C':     "'0'",              # Hardcoded zero
    # 6.1-6.4B are comments only (AU Questionnaire) — no SQL
    '6.1':      "'N/A — AU Questionnaire'",
    '6.2':      "'N/A — AU Questionnaire'",
    '6.3':      "'N/A — AU Questionnaire'",
    '6.4A':     "'N/A — AU Questionnaire'",
    '6.4B':     "'N/A — AU Questionnaire'",
    # Centralized
    '3.19':     "'Not Applicable'",
    '3.20':     "'0'",
    '3.21':     "'0'",
    '3.22':     "'0'",
    # ABAC
    'ABAC1.1':  "'Not Available'",   # SD4 is N/A
    'ABAC1.3':  "'Not Available'",   # SD4 is N/A
    'ABAC3.1':  "'Not Available'",   # 3.2 is N/A
    'ABAC3.2':  "'Not Available'",   # 3.10 is N/A
}

print('Hardcoded / N/A Metrics:')
print('=' * 50)
for metric, value in hardcoded.items():
    result = spark.sql(f"SELECT {value} AS val").collect()[0][0]
    print(f'  {metric:12s} = {result}')

---
## Results Summary
All metrics collected above.

In [ ]:
# ============================================================
# Final Results Summary
# ============================================================
print('=' * 60)
print('TDI 101523 L&H — Optimized Results Summary')
print('=' * 60)

print('\n--- Segment ---')
print(f'{"SD1":15s} (computed above)')
print(f'{"SD3":15s} (computed above)')
print(f'{"CDE 1.6":15s} (computed above)')
print(f'{"CDE 1.7":15s} (computed above)')
print(f'{"SD4":15s} Not Available')
print(f'{"SD5":15s} Not Available')
print(f'{"CDE 4.1A":15s} (computed above)')
print(f'{"CDE 4.1B":15s} (computed above)')
print(f'{"CDE 4.1C":15s} 0')
print(f'{"CDE 5.1-5.9":15s} (computed above)')
print(f'{"CDE 6.1-6.4B":15s} AU Questionnaire')
print(f'{"CDE 6.5A":15s} (computed above)')
print(f'{"CDE 6.5B":15s} (computed above)')

print('\n--- Transaction ---')
print(f'{"CDE 3.1-3.8":15s} (computed above)')
print(f'{"CDE 3.9-3.16":15s} (computed above)')

print('\n--- Centralized ---')
print(f'{"CDE 1.1":15s} (computed above)')
print(f'{"CDE 1.2":15s} (computed above)')
print(f'{"CDE 1.3":15s} (computed above)')
print(f'{"CDE 1.4":15s} (computed above)')
print(f'{"CDE 1.5":15s} (computed above)')
print(f'{"SD2":15s} (computed above)')
print(f'{"CDE 1.8":15s} (computed above)')
print(f'{"CDE 1.9":15s} (computed above)')
print(f'{"SD6":15s} (computed above)')
print(f'{"CDE 3.17":15s} (computed above)')
print(f'{"CDE 3.18":15s} (computed above)')
print(f'{"CDE 3.19":15s} Not Applicable')
print(f'{"CDE 3.20-3.22":15s} 0')

print('\n--- ABAC ---')
print(f'{"ABAC1.1":15s} Not Available')
print(f'{"ABAC1.2":15s} (computed above)')
print(f'{"ABAC1.3":15s} Not Available')
print(f'{"ABAC5.1":15s} (computed above)')
print(f'{"ABAC3.1":15s} Not Available')
print(f'{"ABAC3.2":15s} Not Available')

---
## Cleanup (Optional)
Uncache temporary views to free memory.

In [ ]:
# ============================================================
# Uncache all temp views
# ============================================================
cached_views = [
    'lh_base', 'lh_country', 'lh_centralized', 'lh_branches',
]
for v in cached_views:
    try:
        spark.sql(f'UNCACHE TABLE {v}')
        print(f'Uncached {v}')
    except:
        pass
print('\nCleanup complete.')